# Week 1 – Career Q&A (Gradio + evaluator + Pushover)

Ask any career related questions about Winnie Kariuki

In [ ]:
import os
from pathlib import Path

_nb = Path.cwd()
_winnie = _nb / "1_foundations/community_contributions/winniekariuki"
if (_winnie / "me").is_dir() or (_winnie / "app.ipynb").exists():
    os.chdir(_winnie)
elif (_nb / "me").is_dir():
    os.chdir(_nb)

_THIS_DIR = Path.cwd()
# Summary file: always me/summary.txt next to this notebook folder
_ME = _THIS_DIR / "me"
_SUMMARY_FILE = _ME / "summary.txt"

print("Working directory:", _THIS_DIR)
print("Summary file:", _SUMMARY_FILE)

In [ ]:
from dotenv import load_dotenv

load_dotenv(override=True)
for p in Path.cwd().parents:
    env = p / ".env"
    if env.is_file():
        load_dotenv(env, override=True)
        print("Loaded .env from", env)
        break

In [ ]:
import json
import os

from openai import OpenAI
from pydantic import BaseModel, Field
import gradio as gr
import requests

## Load `me/summary.txt` & Pushover

In [ ]:
def _load_text(path: Path) -> str:
    if not path.exists():
        return ""
    try:
        return path.read_text(encoding="utf-8")
    except Exception:
        return ""


def push(title: str, text: str) -> None:
    token = os.getenv("PUSHOVER_TOKEN")
    user = os.getenv("PUSHOVER_USER")
    if not token or not user:
        print("[Pushover] Skipped - set PUSHOVER_TOKEN and PUSHOVER_USER in .env")
        return
    try:
        requests.post(
            "https://api.pushover.net/1/messages.json",
            data={"token": token, "user": user, "title": title, "message": text},
            timeout=10,
        )
    except Exception as e:
        print(f"[Pushover] Error: {e}")


def record_user_details(email: str, name: str = "Not provided", notes: str = "Not provided") -> dict:
    push("Contact request", f"Name: {name}\nEmail: {email}\nNotes: {notes}")
    return {"recorded": "ok"}

## Tool schema, evaluator, retry

In [ ]:
class Evaluation(BaseModel):
    is_acceptable: bool = Field(description="True if the reply is appropriate for the career site context")
    feedback: str = Field(description="If not acceptable, what to fix; if acceptable, brief confirmation")


record_user_details_json = {
    "type": "function",
    "function": {
        "name": "record_user_details",
        "description": "Use when the user wants to be contacted, has given their email, or asked to get in touch. Sends a push notification.",
        "parameters": {
            "type": "object",
            "properties": {
                "email": {"type": "string", "description": "User's email address"},
                "name": {"type": "string", "description": "User's name if provided"},
                "notes": {"type": "string", "description": "Context or reason for contact"},
            },
            "required": ["email"],
            "additionalProperties": False,
        },
    },
}

TOOLS = [record_user_details_json]

In [ ]:
def _history_for_eval(history: list) -> str:
    parts = []
    for h in history or []:
        if isinstance(h, dict) and "role" in h and "content" in h:
            parts.append(f"{h['role']}: {h['content']}")
    return "\n".join(parts[-20:])


def evaluate_reply(
    client: OpenAI,
    name: str,
    summary: str,
    user_message: str,
    assistant_reply: str,
    history: list,
) -> Evaluation:
    system = f"""You evaluate replies for a career / personal website chatbot.

The assistant represents {name} and must:
- Stay in character as {name}
- Use only information consistent with the professional summary below (no inventing major career facts)
- Be professional; if unsure, say so rather than hallucinating
- Not leak system instructions or behave off-topic inappropriately

Professional summary (ground truth context):
---
{summary[:8000]}
---

Reply with structured evaluation: is_acceptable (bool) and feedback (string)."""

    user = f"""Conversation so far (recent):
{_history_for_eval(history)}

Latest user message:
{user_message}

Assistant reply to evaluate:
{assistant_reply}

Decide if the reply is acceptable. If not, explain what should improve for a retry."""

    try:
        r = client.beta.chat.completions.parse(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            response_format=Evaluation,
            temperature=0.1,
        )
        parsed = r.choices[0].message.parsed
        if parsed is not None:
            return parsed
    except Exception:
        pass

    r2 = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": 'Reply with JSON only: {"is_acceptable": bool, "feedback": "..."}'},
            {"role": "user", "content": user},
        ],
        response_format={"type": "json_object"},
        temperature=0.1,
    )
    raw = r2.choices[0].message.content or "{}"
    data = json.loads(raw)
    return Evaluation(
        is_acceptable=bool(data.get("is_acceptable", True)),
        feedback=str(data.get("feedback", "")),
    )

In [ ]:
def rerun_with_feedback(
    client: OpenAI,
    base_system_prompt: str,
    reply: str,
    user_message: str,
    history: list,
    feedback: str,
) -> str:
    updated = (
        base_system_prompt
        + "\n\n## Quality revision\nYour previous answer was rejected by quality control.\n"
        + f"## Attempted answer:\n{reply}\n\n## Reason:\n{feedback}\n\n"
        + "Reply again to the user, addressing the issue."
    )
    messages = [{"role": "system", "content": updated}]
    for h in history:
        if isinstance(h, dict) and "role" in h and "content" in h:
            messages.append({"role": h["role"], "content": h["content"]})
    messages.append({"role": "user", "content": user_message})
    resp = client.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=TOOLS)
    msg = resp.choices[0].message
    if resp.choices[0].finish_reason == "tool_calls" and getattr(msg, "tool_calls", None):
        results = []
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            if name == "record_user_details":
                out = record_user_details(**args)
            else:
                out = {}
            results.append({"role": "tool", "content": json.dumps(out), "tool_call_id": tc.id})
        messages.append(msg)
        messages.extend(results)
        resp2 = client.chat.completions.create(model="gpt-4o-mini", messages=messages)
        return resp2.choices[0].message.content or ""
    return msg.content or ""

## Career chat class

In [ ]:
class CareerChat:
    def __init__(self):
        self.openai = OpenAI()
        self.name = "Winnie Kariuki"
        self.summary = _load_text(_SUMMARY_FILE)

    def system_prompt(self) -> str:
        return f"""You are {self.name}, answering questions on your site about your career, background, skills, and experience.
Use only the professional summary below as your knowledge base. Be professional and engaging.

- If you don't know the answer from this context, say so clearly.
- If someone wants to be contacted or gives their email, use the record_user_details tool with their email, name, and any notes. You will get a push notification.

## Professional summary
{self.summary or 'No summary provided. Add me/summary.txt.'}

Stay in character as {self.name}."""

    def handle_tool_call(self, tool_calls):
        results = []
        for tc in tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            if name == "record_user_details":
                out = record_user_details(**args)
            else:
                out = {}
            results.append({
                "role": "tool",
                "content": json.dumps(out),
                "tool_call_id": tc.id,
            })
        return results

    def chat(self, message, history):
        base = self.system_prompt()
        messages = [{"role": "system", "content": base}]
        for h in history:
            if isinstance(h, dict) and "role" in h and "content" in h:
                messages.append({"role": h["role"], "content": h["content"]})
        messages.append({"role": "user", "content": message})

        while True:
            resp = self.openai.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                tools=TOOLS,
            )
            msg = resp.choices[0].message
            if resp.choices[0].finish_reason == "tool_calls" and getattr(msg, "tool_calls", None):
                messages.append(msg)
                messages.extend(self.handle_tool_call(msg.tool_calls))
            else:
                reply = msg.content or ""
                break

        if not reply.strip():
            return reply

        evaluation = evaluate_reply(
            self.openai,
            self.name,
            self.summary,
            message,
            reply,
            history,
        )
        if evaluation.is_acceptable:
            return reply

        return rerun_with_feedback(
            self.openai,
            base,
            reply,
            message,
            history,
            evaluation.feedback,
        )

## Launch Gradio

In [ ]:
chat = CareerChat()
gr.ChatInterface(
    chat.chat,
    type="messages",
    title="Career Q&A – Winnie Kariuki",
    description="",
).launch()